In [17]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import gc
from sklearn.preprocessing import LabelEncoder

pd.set_option('display.max_columns', None)


In [18]:
data_path = '/Users/jisuhyeon/Downloads'

df_clean = pd.read_csv(f'{data_path}/df_preprocessed.csv', parse_dates=['timestamp'])

print(df_clean.shape)
print(df_clean.dtypes)
df_clean.head()


(20199231, 18)
building_id                    int64
meter                          int64
timestamp             datetime64[ns]
meter_reading                float64
site_id                        int64
primary_use                   object
square_feet                    int64
air_temperature              float64
cloud_coverage               float64
dew_temperature              float64
precip_depth_1_hr            float64
sea_level_pressure           float64
wind_direction               float64
wind_speed                   float64
time_diff                     object
building_age                 float64
has_floor_count                int64
log_meter_reading            float64
dtype: object


,building_id,meter,timestamp,meter_reading,site_id,primary_use,square_feet,air_temperature,cloud_coverage,dew_temperature,precip_depth_1_hr,sea_level_pressure,wind_direction,wind_speed,time_diff,building_age,has_floor_count,log_meter_reading
0,0,0,2016-01-01,0.0,0,Education,7432,25.0,6.0,20.0,0.0,1019.7,0.0,0.0,0,8.0,0,0.0
1,1,0,2016-01-01,0.0,0,Education,2720,25.0,6.0,20.0,0.0,1019.7,0.0,0.0,0,12.0,0,0.0
2,2,0,2016-01-01,0.0,0,Education,5376,25.0,6.0,20.0,0.0,1019.7,0.0,0.0,0,25.0,0,0.0
3,3,0,2016-01-01,0.0,0,Education,23685,25.0,6.0,20.0,0.0,1019.7,0.0,0.0,0,14.0,0,0.0
4,4,0,2016-01-01,0.0,0,Education,116607,25.0,6.0,20.0,0.0,1019.7,0.0,0.0,0,41.0,0,0.0


In [19]:
# 1. 결측치 없는지 재확인 (전처리 끝난 파일이니 0이어야 정상)
print(df_clean.isnull().sum().sum())

# 2. timestamp가 진짜 datetime으로 잘 들어왔는지
print(df_clean['timestamp'].dtype)  # datetime64[ns] 나와야 함

# 3. 메모리 사용량 확인
print(df_clean.memory_usage(deep=True).sum() / 1024**2, 'MB')

0
datetime64[ns]
5186.840823173523 MB


In [20]:
def reduce_memory(df):
    for col in df.columns:
        col_type = df[col].dtype
        if col_type != object and 'datetime' not in str(col_type):
            c_min = df[col].min()
            c_max = df[col].max()
            if str(col_type)[:3] == 'int':
                if c_min > np.iinfo(np.int8).min and c_max < np.iinfo(np.int8).max:
                    df[col] = df[col].astype(np.int8)
                elif c_min > np.iinfo(np.int16).min and c_max < np.iinfo(np.int16).max:
                    df[col] = df[col].astype(np.int16)
                elif c_min > np.iinfo(np.int32).min and c_max < np.iinfo(np.int32).max:
                    df[col] = df[col].astype(np.int32)
            else:
                if c_min > np.finfo(np.float32).min and c_max < np.finfo(np.float32).max:
                    df[col] = df[col].astype(np.float32)
    return df

df_clean = reduce_memory(df_clean)
print(df_clean.memory_usage(deep=True).sum() / 1024**2, 'MB')  # 줄어든 용량 확인

3819.1331882476807 MB


In [21]:
df_clean['primary_use'] = df_clean['primary_use'].astype('category')

In [22]:
#  사전 준비 - train/valid 분리
df_clean = df_clean.sort_values(['building_id', 'meter', 'timestamp']).reset_index(drop=True)

train_mask = df_clean['timestamp'] < '2016-11-01'
valid_mask = df_clean['timestamp'] >= '2016-11-01'

In [23]:
df_clean = df_clean.sort_values(['building_id', 'meter', 'timestamp']).reset_index(drop=True)

train_df = df_clean[df_clean['timestamp'] < '2016-11-01'].copy()
valid_df = df_clean[df_clean['timestamp'] >= '2016-11-01'].copy()

In [24]:
# 1. 시간 feature
df_clean['hour'] = df_clean['timestamp'].dt.hour
df_clean['day'] = df_clean['timestamp'].dt.day
df_clean['weekday'] = df_clean['timestamp'].dt.dayofweek
df_clean['month'] = df_clean['timestamp'].dt.month
df_clean['is_weekend'] = (df_clean['weekday'] >= 5).astype(int)

In [25]:
# 2. Cyclical encoding
df_clean['hour_sin'] = np.sin(2 * np.pi * df_clean['hour'] / 24)
df_clean['hour_cos'] = np.cos(2 * np.pi * df_clean['hour'] / 24)

df_clean['month_sin'] = np.sin(2 * np.pi * df_clean['month'] / 12)
df_clean['month_cos'] = np.cos(2 * np.pi * df_clean['month'] / 12)

# weekday도 순환 구조라 추가 추천 (월요일-일요일 경계도 이어져야 하니까)
df_clean['weekday_sin'] = np.sin(2 * np.pi * df_clean['weekday'] / 7)
df_clean['weekday_cos'] = np.cos(2 * np.pi * df_clean['weekday'] / 7)

In [26]:
# 3. Lag feature
grouped = df_clean.groupby(['building_id', 'meter'])['log_meter_reading']

df_clean['lag_1'] = grouped.shift(1)
df_clean['lag_2'] = grouped.shift(2)
df_clean['lag_3'] = grouped.shift(3)

df_clean['lag_24'] = grouped.shift(24)
df_clean['lag_48'] = grouped.shift(48)
df_clean['lag_72'] = grouped.shift(72)
df_clean['lag_168'] = grouped.shift(168)

In [27]:
# # 4. Rolling statistics
# grouped = df_clean.groupby(['building_id', 'meter'])['log_meter_reading']

# df_clean['rolling_mean_24'] = grouped.shift(1).groupby([df_clean['building_id'], df_clean['meter']]).rolling(24).mean().reset_index(level=[0,1], drop=True)

# # 1단계: shift(1)부터 별도 컬럼으로 생성 (인덱스 원본 그대로 유지됨)
# df_clean['log_meter_reading_shift1'] = df_clean.groupby(['building_id', 'meter'])['log_meter_reading'].shift(1)

# # 2단계: 그 shift된 값 기준으로 rolling (이것도 인덱스 그대로 유지되니 reset_index 불필요)
# df_clean['rolling_mean_24'] = df_clean.groupby(['building_id', 'meter'])['log_meter_reading_shift1'].transform(lambda x: x.rolling(24).mean())
# df_clean['rolling_std_24'] = df_clean.groupby(['building_id', 'meter'])['log_meter_reading_shift1'].transform(lambda x: x.rolling(24).std())

# # 중간 컬럼 정리 (필요 없으면 삭제)
# df_clean = df_clean.drop(columns=['log_meter_reading_shift1'])

In [28]:
# 4. Rolling statistics
df_clean['lag_1_base'] = grouped.shift(1)

df_clean['rolling_mean_6']  = df_clean.groupby(['building_id','meter'])['lag_1_base'].transform(lambda x: x.rolling(6).mean())
df_clean['rolling_mean_12'] = df_clean.groupby(['building_id','meter'])['lag_1_base'].transform(lambda x: x.rolling(12).mean())
df_clean['rolling_mean_24'] = df_clean.groupby(['building_id','meter'])['lag_1_base'].transform(lambda x: x.rolling(24).mean())
df_clean['rolling_mean_72'] = df_clean.groupby(['building_id','meter'])['lag_1_base'].transform(lambda x: x.rolling(72).mean())

df_clean['rolling_std_24']  = df_clean.groupby(['building_id','meter'])['lag_1_base'].transform(lambda x: x.rolling(24).std())

df_clean.drop(columns=['lag_1_base'], inplace=True)

In [29]:
# Diff feature (성능 핵심)
df_clean['diff_1'] = df_clean['lag_1'] - df_clean['lag_2']
df_clean['diff_24'] = df_clean['lag_24'] - df_clean['lag_48']



In [30]:
# # 5. Building feature
# df_clean['log_square_feet'] = np.log1p(df_clean['square_feet'])

# le_primary_use = LabelEncoder()
# le_primary_use.fit(df_clean.loc[train_mask, 'primary_use'])

# df_clean['primary_use_encoded'] = le_primary_use.fit_transform(df_clean['primary_use'])

# # year_built은 이미 building_age로 파생해뒀으니 그대로 사용
# # building stats - train 기준으로만 계산 (leakage 방지, 지난번 얘기했던 부분)
# building_stats = (
#     df_clean.loc[train_mask]
#     .groupby(['building_id', 'meter'])['log_meter_reading']
#     .agg(['mean', 'std']).add_prefix('building_')
#     .reset_index()
# )
# df_clean = df_clean.merge(building_stats, on=['building_id', 'meter'], how='left')

# # valid에만 있고 train에 없는 building_id/meter 조합은 NaN 발생 가능 → 전체 평균으로 대체
# df_clean['building_mean'] = df_clean['building_mean'].fillna(df_clean.loc[train_mask, 'log_meter_reading'].mean())
# df_clean['building_std'] = df_clean['building_std'].fillna(df_clean.loc[train_mask, 'log_meter_reading'].std())

In [31]:
# 5. Building feature

# (1) 크기 변환
df_clean['log_square_feet'] = np.log1p(df_clean['square_feet'])


# (2) Label Encoding (leakage 방지 버전)
le_primary_use = LabelEncoder()
le_primary_use.fit(df_clean.loc[train_mask, 'primary_use'])  # train 기준 fit

df_clean['primary_use_encoded'] = le_primary_use.transform(df_clean['primary_use'])


# (3) building stats (train 기준)
building_stats = (
    df_clean.loc[train_mask]
    .groupby(['building_id', 'meter'])['log_meter_reading']
    .agg(['mean', 'std'])
    .rename(columns={'mean': 'building_mean', 'std': 'building_std'})
    .reset_index()
)

df_clean = df_clean.merge(building_stats, on=['building_id', 'meter'], how='left')


# (4) fallback 처리 (train global 기준)
global_mean = df_clean.loc[train_mask, 'log_meter_reading'].mean()
global_std  = df_clean.loc[train_mask, 'log_meter_reading'].std()

df_clean['building_mean'] = df_clean['building_mean'].fillna(global_mean)
df_clean['building_std']  = df_clean['building_std'].fillna(global_std)


# (5) ⭐ 핵심 추가 feature (성능 상승 포인트)
df_clean['meter_normalized'] = df_clean['log_meter_reading'] - df_clean['building_mean']

In [32]:
# 6.CDD/HDD
base_temp = 18.3  # 섭씨 기준 (화씨면 65)

df_clean['CDD'] = (df_clean['air_temperature'] - base_temp).clip(lower=0)
df_clean['HDD'] = (base_temp - df_clean['air_temperature']).clip(lower=0)

In [33]:
# 7. Weather interaction

# temp x humidity (dew_temperature를 습도 proxy로 사용)
df_clean['temp_x_dew'] = df_clean['air_temperature'] * df_clean['dew_temperature']
df_clean['temp_diff'] = df_clean['air_temperature'] - df_clean['dew_temperature']

df_clean['humidity_like'] = df_clean['dew_temperature'] / (df_clean['air_temperature'] + 1)
# temp x building type - 지난번 얘기했듯 곱셈보다 primary_use별 평균 온도 반응을 보는 게 나을 수 있음
# 대신 group별 평균으로 반영 (LightGBM이 카테고리+수치 조합을 알아서 분리하긴 하지만, 명시적으로 넣고 싶다면)

# interaction features
df_clean['temp_x_hour'] = df_clean['air_temperature'] * df_clean['hour']
df_clean['temp_x_primary_use'] = df_clean['air_temperature'] * df_clean['primary_use_encoded']

In [34]:
# 최종 확인

# lag/rolling으로 생긴 결측치는 자연스러운 것 (앞부분 168시간 이내 데이터)
new_feature_cols = ['hour_sin','hour_cos','month_sin','month_cos','weekday_sin','weekday_cos',
                     'lag_1','lag_24','lag_168','rolling_mean_24','rolling_std_24',
                     'log_square_feet','primary_use_encoded','building_mean','building_std',
                     'CDD','HDD','temp_x_dew','temp_x_primary_use']

print(df_clean[new_feature_cols].isnull().sum())
print(df_clean.shape)

hour_sin                    0
hour_cos                    0
month_sin                   0
month_cos                   0
weekday_sin                 0
weekday_cos                 0
lag_1                    2378
lag_24                  57072
lag_168                399504
rolling_mean_24         57072
rolling_std_24          57072
log_square_feet             0
primary_use_encoded         0
building_mean               0
building_std                0
CDD                         0
HDD                         0
temp_x_dew                  0
temp_x_primary_use          0
dtype: int64
(20199231, 55)


In [35]:

features = [
    'building_id','site_id','meter',

    'hour','weekday','month','is_weekend',
    'hour_sin','hour_cos','month_sin','month_cos','weekday_sin','weekday_cos',

    'square_feet','log_square_feet',
    'primary_use_encoded','building_age',

    # lag
    'lag_1','lag_2','lag_3',
    'lag_24','lag_48','lag_72','lag_168',

    # rolling
    'rolling_mean_6','rolling_mean_12','rolling_mean_24','rolling_mean_72',
    'rolling_std_24',

    # diff
    'diff_1','diff_24',

    # building stats
    'building_mean','building_std','meter_normalized',

    # weather
    'air_temperature','dew_temperature','temp_x_dew','temp_diff','humidity_like',

    # interaction
    'temp_x_hour','temp_x_primary_use'
]

target = 'log_meter_reading'

In [37]:
data_path = '/Users/jisuhyeon/Downloads'
df_clean.to_csv(f'{data_path}/df_features.csv', index=False)
print(f'저장 완료: {df_clean.shape}')

저장 완료: (20199231, 55)
